# 02 · DMS mutational heatmaps

Visualises the effect of every point mutation on the interaction with pRb
(report, section 1: *Mutational characterisation through heatmaps*).

* **x axis:** WT peptide sequence (positions 2–15 carry mutations).
* **y axis:** amino acid introduced by the substitution.
* **colour:** ΔZ-score = Z_mut − Z_WT. Blue = loss of interaction, red = gain.
* The LxCxE core positions (5, 7 and 9) are highlighted in red.

**Input:** `data/processed/df_tp.csv`, `data/processed/df_dms.csv` (from notebook 01).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Project paths
#   * Locally: start Jupyter from the `notebooks/` folder.
#   * Google Colab: Google Drive is mounted; edit the folder name below so it
#     points to the project folder stored in your Drive.
#   * Anywhere: set the PROJECT_DIR environment variable to override.
# ---------------------------------------------------------------------------
try:
    from google.colab import drive

    drive.mount("/content/drive")
    DEFAULT_PROJECT_DIR = Path("/content/drive/MyDrive/plm-embeddings-lxcxe-prb")
except ImportError:
    DEFAULT_PROJECT_DIR = Path.cwd().parent

PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", DEFAULT_PROJECT_DIR))
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"
for folder in (PROCESSED_DIR, RESULTS_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def save_figure(name):
    """Save the current matplotlib figure into figures/ (call before plt.show())."""
    plt.savefig(FIGURES_DIR / name, dpi=300, bbox_inches="tight")

import seaborn as sns

## 1. Load data

In [ ]:
df_tp = pd.read_csv(PROCESSED_DIR / "df_tp.csv")
df_dms = pd.read_csv(PROCESSED_DIR / "df_dms.csv")

PROTEINS = df_tp["protein"].tolist()
WT_PEPTIDE = df_tp.set_index("protein")["peptide"].to_dict()
WT_SCORES = df_tp.set_index("protein")[["score_pm_rb", "score_pw_rb"]]

AA_ORDER = list("HKRDENQSTCMAILVFWYGP")  # y-axis order
CORE_POSITIONS = (5, 7, 9)  # L, C and E of the LxCxE motif

print(df_dms.groupby("protein", sort=False).size())

## 2. Plotting function

In [ ]:
def build_dms_matrix(protein, value_col):
    """Matrix (mutant amino acid x position) of ΔZ values for one protein."""
    sub = df_dms[df_dms["protein"] == protein]
    matrix = sub.pivot(index="mut_aa", columns="position", values=value_col)
    return matrix.reindex(index=AA_ORDER)


def plot_dms_heatmaps(protein, vmin=None, vmax=None, save=True):
    """Side-by-side ΔZ heatmaps (Pm-Rb and Pw-Rb constructs) for one protein.

    Set vmin/vmax (e.g. -5 and 1) to clip the colour scale; by default it is
    scaled to the data and centred at 0.
    """
    panels = [
        ("Pm-Rb", "delta_pm_rb", "score_pm_rb"),
        ("Pw-Rb", "delta_pw_rb", "score_pw_rb"),
    ]
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    for ax, (label, delta_col, score_col) in zip(axes, panels):
        matrix = build_dms_matrix(protein, delta_col)
        matrix = matrix.dropna(axis=1, how="all")  # keep mutated positions only
        sns.heatmap(
            matrix, ax=ax, cmap="vlag", center=0, vmin=vmin, vmax=vmax,
            linewidths=0.3, linecolor="whitesmoke", cbar_kws={"label": "ΔZ-score"},
        )
        ax.xaxis.tick_top()  # must come before the tick labels are styled
        ax.set_xticklabels(
            [f"{WT_PEPTIDE[protein][pos - 1]}\n{pos}" for pos in matrix.columns], rotation=0
        )
        for tick, pos in zip(ax.get_xticklabels(), matrix.columns):
            if pos in CORE_POSITIONS:
                tick.set_color("red")
                tick.set_fontweight("bold")
        ax.set_xlabel("")
        ax.set_ylabel("Substituted amino acid")
        wt_score = WT_SCORES.loc[protein, score_col]
        ax.set_title(f"{label} ({protein})\nWT Z-score = {wt_score:.3f}", fontsize=14, pad=10)
    plt.tight_layout()
    if save:
        save_figure(f"dms_heatmap_{protein}.png")
    plt.show()

## 3. Heatmaps per protein

Viral peptides: `VE7_HPV16`, `E1A_ADE05_LxCxE`. Human peptides: `ARI4A_HUMAN`, `KDM5A_HUMAN`, `HDAC1_HUMAN`.

In [ ]:
for protein in PROTEINS:
    plot_dms_heatmaps(protein)

### Optional: clipped colour scale

A fixed range makes small effects easier to see when a few mutations produce very large losses.

In [ ]:
plot_dms_heatmaps("ARI4A_HUMAN", vmin=-5, vmax=1, save=False)